In [2]:
import pandas as pd

df = pd.read_csv("/Users/thaonguyen/Downloads/lol_prj/data/2020-spring-match-data-OraclesElixir-2020-05-15.csv")

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15312 entries, 0 to 15311
Data columns (total 99 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   gameid                   15312 non-null  object 
 1   url                      15312 non-null  object 
 2   league                   15312 non-null  object 
 3   split                    15312 non-null  object 
 4   date                     15312 non-null  object 
 5   game                     15312 non-null  int64  
 6   patch                    15312 non-null  float64
 7   playerid                 15312 non-null  int64  
 8   side                     15312 non-null  object 
 9   position                 15312 non-null  object 
 10  player                   12760 non-null  object 
 11  team                     15312 non-null  object 
 12  champion                 12760 non-null  object 
 13  ban1                     15060 non-null  object 
 14  ban2                  

In [3]:
# Count missing values for each column
missing_count = df.isna().sum()

# Calculate missing percentage
missing_pct = df.isna().mean() * 100

# Create a summary table
missing_summary = pd.DataFrame({
    "missing_count": missing_count,
    "missing_pct": missing_pct
})

# Show columns with missing values
missing_summary[missing_summary["missing_count"] > 0] \
    .sort_values("missing_pct", ascending=False)

,missing_count,missing_pct
dragons (type unknown),14584,95.245559
mountains,13488,88.087774
oceans,13488,88.087774
infernals,13488,88.087774
opp_elementaldrakes,13488,88.087774
...,...,...
wardskilled,10,0.065308
controlwardsbought,10,0.065308
damagetochampions,10,0.065308
monsterkillsenemyjungle,10,0.065308


In [4]:
df["position"].value_counts(dropna=False)

position
top     2552
jng     2552
mid     2552
bot     2552
sup     2552
team    2552
Name: count, dtype: int64

In [5]:
df.groupby("position")["champion"].apply(
    lambda x: x.isna().sum()
)

position
bot        0
jng        0
mid        0
sup        0
team    2552
top        0
Name: champion, dtype: int64

In [6]:
# Player-level data for Champion Performance
df_players = df[df["position"] != "team"].copy()

# Team-level data
df_teams = df[df["position"] == "team"].copy()

In [7]:
# Check player rows per match
players_per_match = df_players.groupby("gameid").size()

# Check team rows per match
teams_per_match = df_teams.groupby("gameid").size()

print("Player rows per match:")
print(players_per_match.value_counts())

print("\nTeam rows per match:")
print(teams_per_match.value_counts())

Player rows per match:
10    1276
Name: count, dtype: int64

Team rows per match:
2    1276
Name: count, dtype: int64


In [8]:
# Check fully duplicated rows
print("Full duplicates:", df.duplicated().sum())

# Check duplicate player records by match, side and position
duplicate_picks = df_players.duplicated(
    subset=["gameid", "side", "position"]
).sum()

print("Duplicate player records:", duplicate_picks)

# Check duplicate team records
duplicate_teams = df_teams.duplicated(
    subset=["gameid", "side"]
).sum()

print("Duplicate team records:", duplicate_teams)

Full duplicates: 0
Duplicate player records: 0
Duplicate team records: 0


In [9]:
# Check possible result values
print(df_players["result"].value_counts(dropna=False))

# Check missing results
print("Missing results:", df_players["result"].isna().sum())

# Check number of winning players per match
wins_per_match = df_players.groupby("gameid")["result"].sum()

print("\nWinning players per match:")
print(wins_per_match.value_counts())

result
1    6380
0    6380
Name: count, dtype: int64
Missing results: 0

Winning players per match:
result
5    1276
Name: count, dtype: int64


In [10]:
selected_columns = [
    "gameid", "date", "league", "split",
    "champion", "team", "side", "position",
    "result", "kills", "deaths", "assists",
    "totalgold", "damagetochampions"
]

# Check missing values in player records
df_players[selected_columns].isna().sum().sort_values(
    ascending=False
)

damagetochampions    10
gameid                0
date                  0
league                0
split                 0
champion              0
team                  0
side                  0
position              0
result                0
kills                 0
deaths                0
assists               0
totalgold             0
dtype: int64

In [11]:
ban_columns = ["ban1", "ban2", "ban3", "ban4", "ban5"]

# Check missing bans in team rows
df_teams[ban_columns].isna().sum()

ban1    42
ban2    42
ban3    42
ban4    43
ban5    55
dtype: int64

In [12]:
# Number of recorded bans per team
df_teams[ban_columns].notna().sum(axis=1).value_counts().sort_index()

0      42
3       1
4      12
5    2497
Name: count, dtype: int64

In [15]:
df['league'].unique()

array(['LPL', 'LEC', 'LCS.A', 'CBLOL', 'LCS', 'TCL', 'LCK', 'PCS'],
      dtype=object)

In [16]:
df['gameid'].unique()

array(['5655-7249', '5655-7250', '5656-7252', ...,
       'ESPORTSTMNT04/1210002', 'ESPORTSTMNT04/1210004',
       'ESPORTSTMNT04/1210005'], dtype=object)

## Data Profiling — Key Findings

**Dataset:** Oracle's Elixir — Spring 2020

### Dataset Structure
- 15,312 rows and 99 columns
- 1,276 unique matches
- Each match contains 10 player records and 2 team records

### Data Quality
- No duplicate records were identified using the selected uniqueness keys.
- Champion and result values are complete in player-level records.
- Damage to champions is missing in 10 player records.
- 2,497 of 2,552 team records contain all five ban values.
- 42 team records have no recorded bans.

### Data Modeling Implications
- Player-level records will be used for champion pick and performance analysis.
- Team-level records will be used to extract champion bans.
- Missing damage values will remain NULL.
- Missing ban values require careful treatment to avoid misleading ban-rate calculations.

### Next Step
Finalize the dimensional model and design fact and dimension tables.